# 応用編6

V3.3新機能:freeze機能  

このノートブックでは、スマートコントラクト・バージョン3.3であらたに導入される、freeze機能について説明します。  
freeze機能は、ブロックチェーン上の変更可能なオブジェクトを凍結して変更不可にする機能です。  


## 準備

In [1]:
var { domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc } = await import('../lib/notebook-util.mjs');

## 例1：ユーザ・オブジェクトの凍結

ユーザを新規に作成します。

In [2]:
var resp = await rpc.call(adminWallet, 'c1create', { type: 'user', name: 'adv6user01', domain });
console.log(resp);
var uid = resp.value;

{
  txno: 702017,
  txid: 'xY3dvBFKfUsnva4ZiZsTrpwHuXNzHCdW8ALSrATToZDE7',
  status: 'ok',
  value: 'u58328540'
}


本来はここでウォレットアドレスの紐づけなどの設定などを行いますが、説明を簡単にするため省略します。

この時点でユーザは凍結されておらず変更が可能です。例えば、ユーザの名前が変更できます。

In [3]:
var random_user_name = 'adv6u' + Math.floor(Math.random()*999999);
console.log('new name', random_user_name);
var resp = await rpc.call(adminWallet, 'c1update', { id: uid, prop: 'name', value: random_user_name });
console.log(resp);

new name adv6u652159
{
  txno: 702018,
  txid: 'xEyWVoUocamATEvzoMJkYJHwDwGbhaKkdyvTDMpQ4Cn7p',
  status: 'ok',
  value: null
}


c1freezeを使ってユーザを凍結します。

In [4]:
var resp = await rpc.call(adminWallet, 'c1freeze', { id: uid });
console.log(resp);

{
  txno: 702019,
  txid: 'xVnex4ggg7i7JGS8FF3huEGdLQRb2QN3Ff8QgVFNEtDVMB',
  status: 'ok',
  value: 'u58328540'
}


凍結後は、ユーザの名前は変更できなくなります。  
下記のように名前を変更しようとするとエラーとなります。  


In [5]:
var resp = await rpc.call(adminWallet, 'c1update', { id: uid, prop: 'name', value: 'adv6user3' });
console.log(resp);

{
  txno: 702020,
  txid: 'xqzvj3xqhSHuwCPmXxkMospuPWteu7cgmeT8oiHD2Kmmu',
  status: 'denied',
  value: 'frozen'
}


凍結後は、ユーザのその他の属性も変更できなくなっています。  
例えば下記のようにウォレットを追加しようとしてもエラーとなります。  
(凍結されたオブジェクトに対するc1updateはすべてエラーとなります)  


In [6]:
var resp = await rpc.call(adminWallet, 'c1update', { id: uid, prop: 'add wallets', value: ['eRwsqgzjwaakZTiRX4zjdQssHV8GSf'] });
console.log(resp);

{
  txno: 702021,
  txid: 'xPHZ97wssaUSEQt8S4suiUMuE6BB7b4haPBjGhGma3ZVw',
  status: 'denied',
  value: 'frozen'
}


## 例2：コントラクト・オブジェクトの凍結
上と同様にコントラクトも凍結できます。

コントラクトをデプロイします。

In [7]:
var resp = await rpc.call(adminWallet, 'c1create', { type: 'contract', domain });
console.log(resp);
var cnt = resp.value;

{
  txno: 702022,
  txid: 'xDpFpPYj2gbfbC8pDQTX3w5xzFq3YmzuhPnMAGK5gpuzq',
  status: 'ok',
  value: 'c036574890'
}


この時点でコントラクトは凍結されておらずコードの変更が可能です。

In [8]:
var resp = await rpc.call(adminWallet, 'c1update', { id: cnt, prop: 'code', value: 'return "orange"' });
console.log(resp);

{
  txno: 702023,
  txid: 'xjy65edoTNwFhAgbtYmwnpT35aYRZXvqSmdzR3UGJtC3JB',
  status: 'ok',
  value: null
}


実際にコントラクトを実行するとコードが書き換わっていることが分かります。

In [9]:
var resp = await rpc.call(adminWallet, cnt);
console.log(resp);

{
  txno: 702024,
  txid: 'xA37YgEoq8Yc6i2jaPDfxtQQMDReGBKBcsh8WuCiHDdeFB',
  status: 'ok',
  value: 'orange'
}


c1freezeを使ってコントラクトを凍結します。

In [10]:
var resp = await rpc.call(adminWallet, 'c1freeze', { id: cnt });
console.log(resp);

{
  txno: 702025,
  txid: 'xon7n5uWE85nsJVPrMBzcfuJX7ti3ncfu5AnzMop6Em6o',
  status: 'ok',
  value: 'c036574890'
}


凍結後は、コントラクトのコードは変更できなくなります。  
下記のようにコードを変更しようとするとエラーとなります。  


In [11]:
var resp = await rpc.call(adminWallet, 'c1update', { id: cnt, prop: 'code', value: 'return "apple"' });
console.log(resp);

{
  txno: 702026,
  txid: 'xDBLPy8HQDZN4o9hLgybdc9nWnm5oUMujByNLF64kDZnFB',
  status: 'denied',
  value: 'frozen'
}


実際にコントラクトを実行するとコードが書き換わって*いない*ことが分かります。  
(凍結後でもコントラクトを実行することは可能です)  


In [12]:
var resp = await rpc.call(adminWallet, cnt);
console.log(resp);

{
  txno: 702027,
  txid: 'xvmu3mMA2RBcxNoH9PbXxMmmA6gjixbQVSJqJET6SZRsQB',
  status: 'ok',
  value: 'orange'
}


## 例3：グループ・オブジェクトの凍結
上と同様にグループも凍結できます。

グループを作成します。

In [13]:
var resp = await rpc.call(adminWallet, 'c1create', { type: 'group', domain });
console.log(resp);
var gid = resp.value;

{
  txno: 702028,
  txid: 'xtHGcuTL7GFKEWXJTLfH8687cqXFY7reo6yYX5Aw3GTTBB',
  status: 'ok',
  value: 'g70942970'
}


この時点でグループは凍結されておらずメンバの変更が可能です。

In [14]:
var resp = await rpc.call(adminWallet, 'c1update', { id: gid, prop: 'add members', value: [uid, cnt] });
console.log(resp);

{
  txno: 702029,
  txid: 'xHDSth3sGUfVWw996Xdu4qbEarXc7B3i53q4uLcRy8bqy',
  status: 'ok',
  value: null
}


c1freezeを使ってグループを凍結します。

In [15]:
var resp = await rpc.call(adminWallet, 'c1freeze', { id: gid });
console.log(resp);

{
  txno: 702030,
  txid: 'xbqrc9HEQPoUmtEjzo99AhGBvEXGCmAgrU3RVYNZEQjJEB',
  status: 'ok',
  value: 'g70942970'
}


凍結後は、グループのメンバは変更できなくなります。  
下記のように変更しようとするとエラーとなります。  


In [16]:
var resp = await rpc.call(adminWallet, 'c1update', { id: gid, prop: 'delete members', value: [cnt] });
console.log(resp);

{
  txno: 702031,
  txid: 'xBNgRLpYmf9UhGutaY6hYQGAfBdSrLakugBMKoDwsC3y7',
  status: 'denied',
  value: 'frozen'
}


## 例4：凍結されたオブジェクトの削除
凍結されたオブジェクトであっても、削除することは可能です。

例1～例3で作成した、凍結されたオブジェクトを削除します。

In [17]:
var resp = await rpc.call(adminWallet, 'c1terminate', { id: uid });
console.log(resp);
var resp = await rpc.call(adminWallet, 'c1terminate', { id: cnt });
console.log(resp);
var resp = await rpc.call(adminWallet, 'c1terminate', { id: gid });
console.log(resp);


{
  txno: 702032,
  txid: 'x99r27yv9rYvo9KTTwGkawJFMDmQEJcGJ6tfm38bmD36FB',
  status: 'ok',
  value: 'u58328540'
}
{
  txno: 702033,
  txid: 'xUJ9kmCcBgUpv3sJXkSv3VyjS4qvNGxdZhXUf2UQB3iN9',
  status: 'ok',
  value: 'c036574890'
}
{
  txno: 702034,
  txid: 'x6wGvCdAN9tBFmpPeXtHxELHyivDXBtrMu3EjotUw2ZGEB',
  status: 'ok',
  value: 'g70942970'
}


## 例5：削除された凍結オブジェクトの名前の再利用は不可


凍結したオブジェクトの名前は、そのオブジェクトの削除後であっても再利用できません。  
たとえば、例1で作成した凍結オブジェクトの名前(random_user_name)を再利用しようとするとエラーとなります。  


In [18]:
var resp = await rpc.call(adminWallet, 'c1create', { type: 'user', name: random_user_name, domain });
console.log(resp);


{
  txno: 702035,
  txid: 'xBCMhvUatoFbYV5cPdvzYUzr9aurYvqMrcTehjgGUVwGBB',
  status: 'denied',
  value: 'name is frozen'
}
